# nb_mdf_30_silver_to_gold - metadata-driven Gold build

Gold logic is **SQL held in `meta.gold_transform`** (one row per Gold table, executed in `run_order`).
Business teams can review the SQL in the control table / Purview without opening a notebook, and a new Gold
table is onboarded by inserting a row - the same pattern as ingestion.

The demo rebuilds Gold tables in full (small dimensional tables). Talking points for production:
* Large facts -> incremental MERGE driven by Silver `_silver_updated_at`, or **materialized lake views** with DQ constraints.
* Warehouse-in-Gold -> the same SQL runs as T-SQL CTAS/views in a Fabric Warehouse reading the lakehouse via
  three-part names (see `sql/gold_warehouse_option.sql`).

In [ ]:
transform_ids = ""      # comma-separated; empty = all active transforms
parent_run_id = ""
fail_on_error = True

In [ ]:
%run nb_mdf_utils

In [ ]:
run_id = new_run_id("gold")
gt = spark.table(tname("meta", "gold_transform")).where("active = true")
ids = [x.strip() for x in transform_ids.split(",") if x.strip()]
if ids:
    gt = gt.where(F.col("transform_id").isin(ids))
transforms = [r.asDict() for r in gt.orderBy("run_order", "transform_id").collect()]
print(f"{len(transforms)} gold transform(s) - run {run_id}")

results = {}
for t in transforms:
    started = utcnow()
    target = tname("gold", t["gold_table"])
    try:
        df = spark.sql(t["sql_text"])
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
        n = spark.table(target).count()
        log_run(run_id, parent_run_id, t["transform_id"], "gold", "SUCCESS", started, rows_written=n,
                message=f"{target} rebuilt")
        results[t["transform_id"]] = "SUCCESS"
    except Exception as e:
        log_run(run_id, parent_run_id, t["transform_id"], "gold", "FAILED", started,
                message=f"{target}: {type(e).__name__}: {str(e)[:1500]}")
        results[t["transform_id"]] = "FAILED"

display(spark.table(tname("meta", "run_log")).where(F.col("run_id") == run_id)
        .select("object_id", "status", "rows_written", "duration_sec", "message"))

In [ ]:
failed = [k for k, v in results.items() if v == "FAILED"]
if failed and fail_on_error:
    raise Exception(f"Gold build failed for: {failed}. See meta.run_log for run {run_id}")
notebookutils.notebook.exit(json.dumps({"run_id": run_id, "results": results, "failed": failed}))